# 🎸 Justin Muir • Parametric NAM Offsite Cloud GPU Trainer
### Featuring `mrgeneko/parametric-nam` & NAMix — Free Google Colab GPU Workflow

This notebook provides the **official cloud GPU training pipeline** for Justin Muir's Marshall Origin 50 (Modded) tube head.
You do **not** need an NVIDIA GPU on your local workstation! Train your 42 hardware re-amp takes on Google Cloud's free NVIDIA T4 GPU (~35–45 minutes).

--- 

### 💡 How mrgeneko Parametric Validation Works (No `val.wav` Needed):
- **Traditional NAM**: Requires a separate static 3-minute `v1_1_1.wav` validation file because it only models a single knob snapshot.
- **`mrgeneko/parametric-nam`**: Operates on a continuous multi-dimensional knob space! Its dataset builder (`gen_dataset_from_captures.py`) automatically partitions your 42 sweeps + CSV coordinates across the parameter space (90% training / 10% validation holdout). Your 42 takes are **100% complete, fully valid, and ready to train**!

## Step 1: Verify NVIDIA Cloud GPU Acceleration
Make sure your Colab runtime is set to GPU: **Runtime &rarr; Change runtime type &rarr; Hardware accelerator: T4 GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ NVIDIA Cloud GPU Online: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("❌ No GPU detected! Please go to Runtime -> Change runtime type -> select T4 GPU.")

## Step 2: Clone `mrgeneko/parametric-nam` & Install Dependencies

In [ ]:
!git clone https://github.com/mrgeneko/parametric-nam.git
%cd parametric-nam
!pip install -q lightning torch torchaudio soundfile scipy pandas numpy
print("✅ mrgeneko Parametric NAM engine and dependencies installed successfully!")

## Step 3: Upload Your Zipped Captures (`marshall_origin50_captures.zip`)
Click **Choose Files** below and select your `.zip` archive containing your 42 `.wav` captures and `marshall_origin50_params.csv`.

In [ ]:
import os
import glob
import zipfile
from google.colab import files

os.makedirs("./captures", exist_ok=True)
print("📤 Please select your 'marshall_origin50_captures.zip' file to upload:")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall("./captures/")
        print(f"✅ Successfully unzipped {fn} into ./captures/")
    elif fn.endswith(".wav") or fn.endswith(".csv"):
        os.rename(fn, os.path.join("./captures", fn))
        print(f"✅ Staged {fn} into ./captures/")

# Verify staged files
wav_files = glob.glob("./captures/**/*.wav", recursive=True)
csv_files = glob.glob("./captures/**/*.csv", recursive=True)
print(f"📁 Staged {len(wav_files)} WAV files and {len(csv_files)} CSV files ready for dataset building.")

## Step 4: Build Synchronized Multi-Dimensional Dataset
This runs `gen_dataset_from_captures.py` to index the 42 sweeps across the 8 knob dimensions (Depth, Presence, Master, Treble, Mid, Bass, Gain 1, Gain 2) and automatically partitions training vs. validation slices.

In [ ]:
import os
import glob

os.makedirs("./dataset", exist_ok=True)

# Find CSV if nested
csv_path = glob.glob("./captures/**/marshall_origin50_params.csv", recursive=True)
if not csv_path:
    csv_path = glob.glob("./captures/**/*.csv", recursive=True)
target_csv = csv_path[0] if csv_path else "./captures/marshall_origin50_params.csv"

print(f"Using Parameter CSV: {target_csv}")

!python gen_dataset_from_captures.py \
    --captures "./captures/**/*.wav" \
    --csv "{target_csv}" \
    --output ./dataset \
    --gear-make "Marshall" \
    --gear-model "Origin 50 Modded"

!python gen_dataset_from_captures.py --combine ./dataset

print("✅ Synchronized multi-knob dataset created and combined!")

## Step 5: Train the Parametric WaveNet (`SlimmableParametricA2`)
Trains the continuous neural network with CatMLP/FiLM knob conditioning on the Cloud GPU (~35–45 minutes).

In [ ]:
import os

os.makedirs("./checkpoints", exist_ok=True)

!python param_train.py \
    --dataset ./dataset \
    --output ./marshall_origin50.param.nam \
    --checkpoint-dir ./checkpoints \
    --batch-size 32 \
    --widths 4 8

print("🎉 Neural training complete! Model compiled to marshall_origin50.param.nam")

## Step 6: Download Your Compiled `.param.nam` Model
Directly triggers browser download of your standalone parametric model file.

In [ ]:
import os
from google.colab import files

target_model = "./marshall_origin50.param.nam"
if os.path.exists(target_model):
    print(f"✅ Found compiled model: {target_model} ({os.path.getsize(target_model)/1024:.1f} KB)")
    files.download(target_model)
    print("⬇️ Download initiated! Load this model directly into NAMix (VST3/AU) or your ARM Cortex pedal!")
else:
    print("⚠️ Model file not found at expected path. Check training output above.")

## Step 7: How to Play in NAMix
1. Download and install **NAMix** from [github.com/mrgeneko/NAMix](https://github.com/mrgeneko/NAMix) (VST3, AU, or Standalone).
2. Open NAMix in your DAW (or standalone app).
3. Drag and drop `marshall_origin50.param.nam` into NAMix.
4. Enjoy live, continuous, zero-zipper-noise rotary knob control across all 8 parameters of your physical Marshall Origin 50!